# 🏢 Notebook 03: Modelagem Preditiva Específica do Subgrupo B3 (Comercial)
**Disciplina**: Técnicas de IA Aplicadas a Sistemas de Energia  
**Programa**: Mestrado Profissional — IFSC  
**Autor**: Dilson Eijo Rigotti  

---

## 🎯 Objetivo
Concentrar a engenharia de dados, a decomposição estatística (STL) e o treinamento de modelos preditivos exclusivamente sobre o **Subgrupo B3 (Comercial / Baixa Tensão)** em Santa Catarina. Este grupo é o primeiro elegível para migração ao Mercado Livre de Energia em **Novembro de 2027**, conforme estabelecido pela **Lei nº 15.269/2025**.

Reavaliamos qual modelo de IA (SARIMAX vs Holt-Winters vs Gradient Boosting ML) apresenta maior acurácia especificamente para a dinâmica comercial catarinense.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import json

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Carregamento dos Dados do Subgrupo B3 Comercial (1994 - 2026)

In [ ]:
df = pd.read_csv('../data/processed/sc_grupo_b_mensal.csv')
df['data'] = pd.to_datetime(df['data'])

df_b3 = df[df['classe'] == 'Comercial'].sort_values('data').copy()
df_b3.set_index('data', inplace=True)
ts_b3 = df_b3['consumo_mwh'].asfreq('MS').interpolate(method='linear')

print(f"Total de meses B3: {len(ts_b3)} (de {ts_b3.index[0].strftime('%Y-%m')} a {ts_b3.index[-1].strftime('%Y-%m')})")
ts_b3.head()

## 2. Decomposição STL do Consumo Comercial B3
Decomposição aditiva da série temporal $Y_t = T_t + S_t + R_t$ para isolar a taxa de expansão do comércio catarinense.

In [ ]:
stl = STL(ts_b3, seasonal=13, robust=True)
res = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
axes[0].plot(res.observed, color='#1f77b4', linewidth=1.5)
axes[0].set_ylabel('Observado (MWh)')
axes[0].set_title('Decomposição STL - Subgrupo B3 Comercial (SC 1994-2026)', fontsize=13, fontweight='bold')

axes[1].plot(res.trend, color='#ff7f0e', linewidth=2)
axes[1].set_ylabel('Tendência')

axes[2].plot(res.seasonal, color='#2ca02c', linewidth=1.5)
axes[2].set_ylabel('Sazonalidade')

axes[3].plot(res.resid, color='#d62728', marker='.', linestyle='None', alpha=0.5)
axes[3].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[3].set_ylabel('Resíduo')
axes[3].set_xlabel('Ano')

plt.tight_layout()
plt.show()

## 3. Treinamento e Reavaliação Preditiva no Período de Teste (2024 - 2026)
Comparativo entre SARIMAX, Holt-Winters e Gradient Boosting Machine Learning.

In [ ]:
split_date = '2024-01-01'
train = ts_b3[ts_b3.index < split_date]
test = ts_b3[ts_b3.index >= split_date]

# 1. SARIMAX
model_sarimax = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12))
res_sarimax = model_sarimax.fit(disp=False)
pred_sarimax = res_sarimax.predict(start=test.index[0], end=test.index[-1])

# 2. Holt-Winters
model_hw = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12)
res_hw = model_hw.fit()
pred_hw = res_hw.predict(start=test.index[0], end=test.index[-1])

# 3. Gradient Boosting ML
def create_lags(series):
    df_f = pd.DataFrame(index=series.index)
    df_f['y'] = series.values
    df_f['mes'] = df_f.index.month
    df_f['ano'] = df_f.index.year
    for lag in range(1, 13):
        df_f[f'lag_{lag}'] = df_f['y'].shift(lag)
    return df_f.dropna()

df_feat = create_lags(ts_b3)
train_f = df_feat[df_feat.index < split_date]
test_f = df_feat[df_feat.index >= split_date]
X_train, y_train = train_f.drop(columns=['y']), train_f['y']
X_test, y_test = test_f.drop(columns=['y']), test_f['y']

model_gb = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=42)
model_gb.fit(X_train, y_train)
pred_gb = pd.Series(model_gb.predict(X_test), index=y_test.index)

## 4. Tabela Comparativa de Acurácia no Subgrupo B3

In [ ]:
with open('../data/processed/b3_model_metrics.json', 'r', encoding='utf-8') as f:
    b3_metrics = json.load(f)

pd.DataFrame(b3_metrics)

## 5. Projeção de Demanda B3 para o Marco de Novembro/2027 (Abertura Mercado Libre Comercial)

In [ ]:
future_dates = pd.date_range(start='2026-04-01', periods=33, freq='MS')
model_best = SARIMAX(ts_b3, order=(1,1,1), seasonal_order=(1,1,1,12))
res_best = model_best.fit(disp=False)
forecast_b3 = res_best.get_forecast(steps=33)
pred_b3_mean = forecast_b3.predicted_mean
conf_b3 = forecast_b3.conf_int()

plt.figure(figsize=(15, 6))
plt.plot(ts_b3['2020-01-01':].index, ts_b3['2020-01-01':].values, label='Consumo Histórico B3 Comercial (MWh)', color='black', linewidth=1.8)
plt.plot(test.index, pred_gb, label='Gradient Boosting ML (Teste 2024-2026)', color='#2ca02c', linestyle='--', linewidth=2)
plt.plot(future_dates, pred_b3_mean, label='Projeção Futura B3 Comercial (2026-2028)', color='#9467bd', linewidth=2.5)
plt.fill_between(future_dates, conf_b3.iloc[:, 0], conf_b3.iloc[:, 1], color='#9467bd', alpha=0.2, label='Intervalo de Confiança 95%')

plt.axvline(pd.to_datetime('2027-11-01'), color='red', linestyle='--', linewidth=2, label='Novembro/2027: Abertura B3 Comercial (Lei 15.269/2025)')

plt.title('Projeção Preditiva de Consumo (MWh) - Subgrupo B3 Comercial SC', fontsize=14, fontweight='bold')
plt.xlabel('Ano')
plt.ylabel('Consumo Mensal (MWh)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()